# CPG-RL 地形版訓練 Notebook（Go2 · MJX · 平地→斜坡過渡）

在論文標準版（`cpg_rl_paper_colab.ipynb`）基礎上，**加入斜坡地形訓練**（方案 A，見 `docs/cpg_rl_terrain_training_study.md`）。

**與論文版的差異（只動地形相關，CPG/IK/PD/DR/PPO 全部沿用）**：
- **地形**：中央平台 + 分段逐漸變陡的斜坡（上坡 +x：7.5°→15°；下坡 −x：鏡像）。用 **box 幾何**（MJX 最穩，非 hfield）。原無限地板降到 z=−10 當安全底網。
- **reset 隨機朝向**：面 +x（走上坡）或 −x（走下坡）→ 涵蓋上/下坡。
- **curriculum**：地形本身「越走越陡」，靠里程獎勵自然先學緩坡、走穩才推進到陡段（distance-based emergent，不需全域步數）。
- **修好三個「寫死平地」地雷**：身高/跌倒判定改 `rel_h = z − gz(x)`；觸地布林改 `foot_z − gz(foot_x) < 門檻`；g_c 斜坡維持固定。
- **觀測維持 76 維（盲走）**：斜坡靠 obs 既有的重力向量感知，不加 heightmap。

> 地形幾何已用射線量測驗證 == 解析 `gz(x)`；env 已在本機 CPU MJX 冒煙測試（obs=76、上下坡 spawn、reward 有限、done 不誤觸發）。
> ⚠️ 仍未在 GPU 實跑，務必先跑 Smoke test。

## 第 1 步：GPU + 安裝
執行階段 → 變更類型 → GPU。`MUJOCO_GL=egl` 要在 import mujoco 前設好。

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
!pip install -q mujoco mujoco-mjx brax mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

## 第 1.5 步：Go2 模型（用 scene_mjx.xml，位置伺服→apply_pd 改等效 kp=90/kd=3）

In [ ]:
import os, subprocess
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/google-deepmind/mujoco_menagerie.git"], check=True)
SCENE = "mujoco_menagerie/unitree_go2/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

## 第 2 步：論文版 CPG（JAX）— 與論文版完全相同

In [ ]:
import jax.numpy as jnp

MU_MIN, MU_MAX = 1.0, 2.0
OMEGA_MIN, OMEGA_MAX = 0.0, 4.5
A_CONV = 50.0
D_STEP = 0.12
G_C = 0.08              # 擺動離地（固定）
G_P = 0.01
W_COUP = 8.0
N_CPG_SUB = 4
PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])          # trot FL,FR,RL,RR
PHI = PHASE_OFFSET[None, :] - PHASE_OFFSET[:, None]


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4), "theta": PHASE_OFFSET}


def cpg_step(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI
        coup = jnp.sum(rbar[None, :] * jnp.sin(diff), axis=1)
        th = th + (2.0 * jnp.pi * omega + W_COUP * coup) * h
    th = jnp.mod(th, 2.0 * jnp.pi)
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd, "theta": th}


def action_to_cpg_cmd(action):
    a = jnp.tanh(action).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    omega = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, omega


def cpg_foot_offsets(c):
    th = c["theta"]
    fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    dx = -D_STEP * fx * jnp.cos(th)
    dy = D_STEP * fy * jnp.cos(th)
    dz = jnp.where(jnp.sin(th) > 0, G_C * jnp.sin(th), G_P * jnp.sin(th))
    return jnp.stack([dx, dy, dz], axis=-1)


def cpg_to_joint_targets(c, jinvs, home3):
    off = cpg_foot_offsets(c)
    dq = jnp.einsum("kij,kj->ki", jinvs, off)
    q = home3[None, :] + dq
    return q.reshape(12)

_c = cpg_init()
_c = cpg_step(_c, jnp.full(4, 1.8), jnp.full(4, 1.5), jnp.full(4, 2.0), 0.02)
print("cpg ok, theta=", _c["theta"])

## 第 3 步：每腿 3D IK 常數（terrain-independent，用原始場景 home 姿態算）

In [ ]:
import numpy as np, mujoco

LEGS = ["FL", "FR", "RL", "RR"]
HOME3_np = np.array([0.0, 0.9, -1.8])

def leg_ik_consts(xml):
    m = mujoco.MjModel.from_xml_path(xml); d = mujoco.MjData(m)
    f0s, jinvs = [], []
    for k, leg in enumerate(LEGS):
        jb = 7 + 3 * k
        gid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg)
        hip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, leg + "_hip")
        def foot(q3):
            mujoco.mj_resetDataKeyframe(m, d, 0)
            d.qpos[jb:jb + 3] = q3; mujoco.mj_forward(m, d)
            return (d.geom_xpos[gid] - d.xpos[hip]).copy()
        f0 = foot(HOME3_np); e = 1e-3; J = np.zeros((3, 3))
        for j in range(3):
            dq = np.zeros(3); dq[j] = e
            J[:, j] = (foot(HOME3_np + dq) - foot(HOME3_np - dq)) / (2 * e)
        f0s.append(f0); jinvs.append(np.linalg.inv(J))
    return np.array(f0s, np.float32), np.array(jinvs, np.float32)

F0S_np, JINVS_np = leg_ik_consts(SCENE)
print("IK 常數就緒")

## 第 3.5 步：地形（平台 + 分段斜坡）★新

- 折點 `KNOTS_(X,Z)` 定義地面剖面：平台 [−1,1]=0；上坡 +x（7.5°→15°）；下坡 −x 鏡像。
- `build_terrain_model()`：用 MjSpec 把 5 塊 box 依折點鋪成連續斜面，原地板降到 z=−10 安全底網。
- `gz_np/gz_j(x)`：解析地面高度（`interp`），**與 box 幾何同源** → reward/觸地/跌倒判定都靠它換算相對地面高度。
- 已用 `mj_ray` 驗證實際幾何表面 == `gz`（誤差 <0.001）。

In [ ]:
TERR_X0 = 1.0
_dz1 = 2.0 * np.tan(np.radians(7.5))
_dz2 = 3.0 * np.tan(np.radians(15.0))
KNOTS_X = np.array([-6.0, -3.0, -1.0, 1.0, 3.0, 6.0], np.float32)
KNOTS_Z = np.array([-(_dz1 + _dz2), -_dz1, 0.0, 0.0, _dz1, _dz1 + _dz2], np.float32)
TERR_WY, TERR_TH = 3.0, 0.5
KNOTS_X_j, KNOTS_Z_j = jnp.array(KNOTS_X), jnp.array(KNOTS_Z)


def gz_np(x): return np.interp(x, KNOTS_X, KNOTS_Z)
def gz_j(x):  return jnp.interp(x, KNOTS_X_j, KNOTS_Z_j)


def build_terrain_model():
    spec = mujoco.MjSpec.from_file(SCENE)
    floor = next(g for g in spec.geoms if g.name == "floor")
    floor.pos = [0.0, 0.0, -10.0]                       # 安全底網
    for i in range(len(KNOTS_X) - 1):
        xa, za, xb, zb = KNOTS_X[i], KNOTS_Z[i], KNOTS_X[i + 1], KNOTS_Z[i + 1]
        L = float(np.hypot(xb - xa, zb - za)); a = float(np.arctan2(zb - za, xb - xa))
        mx, mz = 0.5 * (xa + xb), 0.5 * (za + zb)
        nx, nz = -np.sin(a), np.cos(a)                  # 頂面外法線
        g = spec.worldbody.add_geom()
        g.name = f"ramp{i}"; g.type = mujoco.mjtGeom.mjGEOM_BOX
        g.size = [L / 2, TERR_WY, TERR_TH]
        g.pos = [mx - nx * TERR_TH, 0.0, mz - nz * TERR_TH]
        g.quat = [np.cos(a / 2), 0.0, -np.sin(a / 2), 0.0]
        g.rgba = [0.55, 0.5, 0.45, 1.0]
    return spec.compile()

# 幾何 vs gz 快速核對（CPU 射線）
_m = build_terrain_model(); _d = mujoco.MjData(_m); mujoco.mj_forward(_m, _d)
_bad = 0
for _x in np.arange(-5.5, 5.6, 0.5):
    if abs(_x) < 0.9: continue                          # 平台上有機器人，略過
    _gid = np.zeros(1, np.int32)
    _dist = mujoco.mj_ray(_m, _d, np.array([_x,0,5.0]), np.array([0,0,-1.0]), None, 1, -1, _gid)
    if abs((5.0 - _dist) - gz_np(_x)) > 0.02: _bad += 1
print("地形折點 z=", np.round(KNOTS_Z, 3), " 幾何/gz 不一致點數=", _bad, "(應為0)")

## 第 4 步：MJX 環境（地形版）

改自論文版，只動地形相關：
- 模型用 `build_terrain_model()`。
- `reset`：spawn 在平台(x=0)，隨機面 +x(上坡)/−x(下坡)。
- `_foot_contact`：`foot_z − gz(foot_x) < 3cm`（相對地面）。
- `step`：獎勵/跌倒用 `rel_h = z − gz(x)`；其餘（r_lin/r_yaw/upright/act_rate/抗推）與論文版相同。

In [ ]:
import functools
from brax.envs.base import Env, State
from mujoco import mjx

CTRL_DT, SIM_DT = 0.02, 0.004
N_FRAMES = int(round(CTRL_DT / SIM_DT))
HOME12 = jnp.array([0.0, 0.9, -1.8] * 4)
HOME3 = jnp.array([0.0, 0.9, -1.8])
KP_NOM, KD_NOM = 90.0, 3.0
KNEE_IDX = [2, 5, 8, 11]
FOOT_CONTACT_H = 0.03
PUSH_EVERY = 100
PUSH_VEL = 0.6


def apply_pd(m, kp=KP_NOM, kd=KD_NOM):
    m.actuator_gainprm[:, 0] = kp
    m.actuator_biasprm[:, 0] = 0.0
    m.actuator_biasprm[:, 1] = -kp
    m.actuator_biasprm[:, 2] = -kd
    fr = np.full(m.nu, 23.7); fr[KNEE_IDX] = 45.43
    m.actuator_forcerange[:, 0] = -fr; m.actuator_forcerange[:, 1] = fr
    m.actuator_forcelimited[:] = 1
    return m


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)
def w2b(quat, v): return _qrot(_qinv(quat), v)


class Go2TerrainCpgEnv(Env):
    def __init__(self, jinvs):
        m = build_terrain_model(); m.opt.timestep = SIM_DT
        m = apply_pd(m)
        self._mj = m
        self.sys = mjx.put_model(m)
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._jinvs = jnp.array(jinvs)
        self._foot_gid = jnp.array(
            [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in LEGS])

    @property
    def observation_size(self): return 76
    @property
    def action_size(self): return 12
    @property
    def backend(self): return "mjx"

    def _sample_cmd(self, rng):
        # 直走：只隨機前進速度，不下橫移/轉向指令 → 走歪就被扣分
        vx = jax.random.uniform(rng, (), minval=0.4, maxval=0.9)
        return jnp.array([vx, 0.0, 0.0])

    def _base(self, data):
        quat = data.qpos[3:7]; gyro = data.qvel[3:6]
        blin = w2b(quat, data.qvel[0:3])
        grav = w2b(quat, jnp.array([0.0, 0.0, -1.0]))
        return quat, gyro, blin, grav

    def _foot_contact(self, data):
        fx = data.geom_xpos[self._foot_gid, 0]
        fz = data.geom_xpos[self._foot_gid, 2]
        return (fz - gz_j(fx) < FOOT_CONTACT_H).astype(jnp.float32)

    def _obs(self, data, info):
        _, gyro, blin, grav = self._base(data)
        c = info["cpg"]
        obs = jnp.concatenate([
            grav, blin, gyro,
            data.qpos[7:19] - HOME12, data.qvel[6:18],
            info["cmd"], info["last_action"], self._foot_contact(data),
            c["rx"], c["rx_d"], c["ry"], c["ry_d"],
            jnp.sin(c["theta"]), jnp.cos(c["theta"]),
        ])
        # 防護：任一環境物理發散的 inf/nan 不得汙染 normalize 統計
        return jnp.nan_to_num(jnp.clip(obs, -50.0, 50.0), nan=0.0)

    def reset(self, rng):
        rng, crng, hrng = jax.random.split(rng, 3)
        downhill = jax.random.bernoulli(hrng, 0.5)
        quat = jnp.where(downhill, jnp.array([0.0, 0.0, 0.0, 1.0]),
                         jnp.array([1.0, 0.0, 0.0, 0.0]))          # 面 -x 下坡 / +x 上坡
        qpos = self._init_q.at[3:7].set(quat)
        data = mjx.make_data(self.sys).replace(qpos=qpos)
        data = mjx.forward(self.sys, data)
        info = {"rng": rng, "cmd": self._sample_cmd(crng),
                "cpg": cpg_init(), "last_action": jnp.zeros(12),
                "step": jnp.zeros((), jnp.int32)}
        obs = self._obs(data, info)
        metrics = {"reward": jnp.zeros(()), "r_lin": jnp.zeros(()),
                   "r_yaw": jnp.zeros(()), "rel_h": jnp.zeros(()),
                   "climb": jnp.zeros(()), "y_drift": jnp.zeros(())}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        mux, muy, omega = action_to_cpg_cmd(action)
        cpg = cpg_step(state.info["cpg"], mux, muy, omega, CTRL_DT)
        q_des = cpg_to_joint_targets(cpg, self._jinvs, HOME3)
        ctrl = jnp.clip(q_des, self._lo, self._hi)

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=ctrl)), None
        data, _ = jax.lax.scan(one, state.pipeline_state, None, N_FRAMES)

        rng, krng = jax.random.split(state.info["rng"])
        step_i = state.info["step"] + 1
        do_push = jnp.mod(step_i, PUSH_EVERY) == 0
        kick = jax.random.uniform(krng, (2,), minval=-PUSH_VEL, maxval=PUSH_VEL)
        qvel = (data.qvel.at[0].add(jnp.where(do_push, kick[0], 0.0))
                          .at[1].add(jnp.where(do_push, kick[1], 0.0)))
        data = data.replace(qvel=qvel)

        info = {**state.info, "cpg": cpg, "last_action": action,
                "rng": rng, "step": step_i}
        obs = self._obs(data, info)
        _, gyro, blin, grav = self._base(data)
        cmd = info["cmd"]
        r_lin = jnp.exp(-((blin[0] - cmd[0]) ** 2 + (blin[1] - cmd[1]) ** 2) / 0.25)
        r_yaw = jnp.exp(-((gyro[2] - cmd[2]) ** 2) / 0.25)
        upright = grav[0] ** 2 + grav[1] ** 2
        gzb = gz_j(data.qpos[0])
        rel_h = data.qpos[2] - gzb                       # ★ 相對地面高度
        height_pen = (rel_h - 0.30) ** 2
        y_drift = data.qpos[1]                           # ★ 世界橫向偏移(spawn y=0)：走歪就大
        y_pen = 1.0 - jnp.exp(-(y_drift ** 2) / 0.10)    # 有界[0,1]：y=0不罰, ~0.5m 飽和
        act_rate = jnp.sum((action - state.info["last_action"]) ** 2)
        reward = (1.5 * r_lin + 1.2 * r_yaw - 1.0 * upright
                  - 0.5 * height_pen - 0.8 * y_pen - 0.05 * act_rate + 0.05)
        done = jnp.where((rel_h < 0.18) | (grav[2] > -0.4), 1.0, 0.0)
        # 有限性防護：物理發散→結束該回合、reward 歸零，杜絕 nan 擴散
        finite = (jnp.isfinite(reward) & jnp.all(jnp.isfinite(data.qpos))
                  & jnp.all(jnp.isfinite(data.qvel)))
        reward = jnp.where(finite, reward, 0.0)
        done = jnp.where(finite, done, 1.0)
        metrics = {"reward": reward, "r_lin": r_lin, "r_yaw": r_yaw,
                   "rel_h": rel_h, "climb": gzb, "y_drift": jnp.abs(y_drift)}
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)

print("terrain env defined")

## 第 4.5 步：Domain Randomization（與論文版相同）

地形為**共享靜態**（不隨機化），故 DR 只randomize 摩擦/PD/質量/負重，寫法與論文版一致。

In [ ]:
_mm = build_terrain_model()
BASE_ID = mujoco.mj_name2id(_mm, mujoco.mjtObj.mjOBJ_BODY, "base")

def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5 = jax.random.split(rng, 5)
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.3, maxval=1.0))
        kp = jax.random.uniform(k2, minval=75.0, maxval=105.0)
        kd = jax.random.uniform(k3, minval=2.0, maxval=4.0)
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.8, maxval=1.2)
        payload = jax.random.uniform(k5, minval=0.0, maxval=8.0)
        body_mass = body_mass.at[BASE_ID].add(payload)
        return geom_friction, gain, bias, body_mass
    gf, gain, bias, bm = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm)
    return sys, in_axes
print("domain_randomize ready")

## Smoke test（開訓練前必跑）

In [ ]:
env = Go2TerrainCpgEnv(JINVS_np)
for seed in [0, 1, 2, 3]:
    s = jax.jit(env.reset)(jax.random.PRNGKey(seed))
    s2 = jax.jit(env.step)(s, jnp.zeros(12))
    face = "下坡(-x)" if float(s.pipeline_state.qpos[6]) > 0.5 else "上坡(+x)"
    print(f"[seed {seed}] obs={s.obs.shape} {face} reward={float(s2.reward):+.3f} "
          f"done={float(s2.done):.0f} rel_h={float(s2.metrics['rel_h']):.3f}")
print("PASSED" if s.obs.shape == (76,) else "CHECK OBS SIZE")

## 第 5 步：Brax PPO 訓練

與論文版相同超參；地形較難，`num_timesteps` 給 1.5e8。OOM 就降 `num_envs`。

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = Go2TerrainCpgEnv(JINVS_np)
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

train_fn = functools.partial(
    ppo.train, num_timesteps=150_000_000, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=domain_randomize, seed=0)

_t0 = time.time(); rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    print(f"step {step:>10,}  reward {r:8.2f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s,_ in rewards], [r for _,r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()

## 看成果：CPU rollout 影片（上坡 + 下坡各一段）

obs 組法與訓練逐項一致（含 gz 相對地面觸地）。spawn 面 +x 走上坡、面 −x 走下坡。

In [ ]:
import mediapy as media

infer = jax.jit(make_inference_fn(params, deterministic=True))

def cpg_init_np():
    return {"rx": np.full(4, 1.5), "rx_d": np.zeros(4),
            "ry": np.full(4, 1.5), "ry_d": np.zeros(4),
            "theta": np.array([0.0, np.pi, np.pi, 0.0])}
PHI_np = np.array([0.0, np.pi, np.pi, 0.0]); PHI_np = PHI_np[None, :] - PHI_np[:, None]

def cpg_step_np(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = (c["rx"].copy(), c["rx_d"].copy(),
                            c["ry"].copy(), c["ry_d"].copy(), c["theta"].copy())
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd += (A_CONV * (A_CONV / 4 * (mux - rx) - rxd)) * h; rx += rxd * h
        ryd += (A_CONV * (A_CONV / 4 * (muy - ry) - ryd)) * h; ry += ryd * h
        rbar = 0.5 * (rx + ry); diff = th[None, :] - th[:, None] - PHI_np
        th = th + (2*np.pi*omega + W_COUP*np.sum(rbar[None,:]*np.sin(diff),1)) * h
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd, "theta": th % (2*np.pi)}

def act_to_cmd_np(a):
    a = np.tanh(a).reshape(4, 3)
    mux = (a[:,0]+1)/2*(MU_MAX-MU_MIN)+MU_MIN; muy = (a[:,1]+1)/2*(MU_MAX-MU_MIN)+MU_MIN
    om = (a[:,2]+1)/2*(OMEGA_MAX-OMEGA_MIN)+OMEGA_MIN
    return mux, muy, om

def targets_np(c, jinvs):
    th = c["theta"]
    fx = 2*(c["rx"]-MU_MIN)/(MU_MAX-MU_MIN)-1; fy = 2*(c["ry"]-MU_MIN)/(MU_MAX-MU_MIN)-1
    dx = -D_STEP*fx*np.cos(th); dy = D_STEP*fy*np.cos(th)
    dz = np.where(np.sin(th) > 0, G_C*np.sin(th), G_P*np.sin(th))
    off = np.stack([dx, dy, dz], -1); q = np.zeros((4, 3))
    for k in range(4): q[k] = HOME3_np + jinvs[k] @ off[k]
    return q.reshape(12)

def qinv(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def qrot(q, v):
    u = q[1:4]; t = 2*np.cross(u, v); return v + q[0]*t + np.cross(u, t)
def w2b_np(q, v): return qrot(qinv(q), v)

def rollout(downhill, n=500):
    m = build_terrain_model(); m.opt.timestep = SIM_DT; m = apply_pd(m)
    d = mujoco.MjData(m); mujoco.mj_resetDataKeyframe(m, d, 0)
    d.qpos[3:7] = [0,0,0,1.0] if downhill else [1.0,0,0,0]     # 面 -x / +x
    mujoco.mj_forward(m, d)
    lo = m.actuator_ctrlrange[:,0]; hi = m.actuator_ctrlrange[:,1]
    foot_gid = [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in LEGS]
    fl = foot_gid[0]; nsub = int(round(CTRL_DT/SIM_DT))
    ren = mujoco.Renderer(m, 480, 640); cam = mujoco.MjvCamera(); mujoco.mjv_defaultFreeCamera(m, cam)
    cmd = np.array([0.6, 0.0, 0.0]); c = cpg_init_np(); last_a = np.zeros(12)
    frames = []; x0 = float(d.qpos[0])
    for i in range(n):
        grav = w2b_np(d.qpos[3:7], np.array([0,0,-1.0])); blin = w2b_np(d.qpos[3:7], d.qvel[0:3])
        fxs = np.array([d.geom_xpos[g][0] for g in foot_gid]); fzs = np.array([d.geom_xpos[g][2] for g in foot_gid])
        contact = ((fzs - gz_np(fxs)) < FOOT_CONTACT_H).astype(np.float32)
        obs = np.concatenate([grav, blin, d.qvel[3:6], d.qpos[7:19]-np.array([0,0.9,-1.8]*4),
                              d.qvel[6:18], cmd, last_a, contact,
                              c["rx"], c["rx_d"], c["ry"], c["ry_d"],
                              np.sin(c["theta"]), np.cos(c["theta"])]).astype(np.float32)
        act = np.array(infer(jnp.asarray(obs), jax.random.PRNGKey(0)))
        mux, muy, om = act_to_cmd_np(act); c = cpg_step_np(c, mux, muy, om, CTRL_DT)
        q_des = targets_np(c, JINVS_np); d.ctrl[:] = np.clip(q_des, lo, hi)
        for _ in range(nsub): mujoco.mj_step(m, d)
        last_a = act
        if i % 2 == 0:
            cam.lookat[:] = d.qpos[:3]; cam.distance = 2.5; cam.elevation = -18; cam.azimuth = 90
            ren.update_scene(d, cam); frames.append(ren.render())
    dist = abs(float(d.qpos[0]) - x0); climb = gz_np(float(d.qpos[0]))
    print(("下坡" if downhill else "上坡"), f"水平位移={dist:.2f}m 末端地面高={climb:+.2f}m 末端rel_h={float(d.qpos[2])-climb:.2f}")
    return frames

print("=== 上坡 ==="); f_up = rollout(False)
print("=== 下坡 ==="); f_dn = rollout(True)
media.show_video(f_up, fps=25)
media.show_video(f_dn, fps=25)

## 第 6 步：存權重下載

In [ ]:
from brax.io import model
model.save_params("cpg_rl_terrain_params.pkl", params)
try:
    from google.colab import files; files.download("cpg_rl_terrain_params.pkl")
except Exception as e:
    print("左側檔案面板右鍵下載 cpg_rl_terrain_params.pkl。", e)

## 帶回本機

本機推論要用地形版：obs 需以 `gz_np` 換算相對地面觸地（同本檔 rollout）。網路設定與本檔一致
（policy 256/256/128、value 256³、normalize、動作12、觀測76）。CPG/IK 與論文版相同。
可仿 `inference/local_infer_paper.py` 另存一支 `local_infer_terrain.py`（把觸地布林改成 gz 相對地面）。